<a href="https://colab.research.google.com/github/Capechusami/Zindi_Financial_Inclusion/blob/0.102627664/Zindi_Financial_Inclusion__2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Install CatBoost if needed, then import everything ─────────────────────
try:
    from catboost import CatBoostClassifier
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'catboost', '-q'], check=True)
    from catboost import CatBoostClassifier

import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import time
import warnings
warnings.filterwarnings('ignore')

try:
    from lightgbm import early_stopping
    LGBM_NEW_API = True
except ImportError:
    LGBM_NEW_API = False

# ── Data path ──────────────────────────────────────────────────────────────
DATA_PATH = './'

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────
train = pd.read_csv(DATA_PATH + 'Train.csv')
test  = pd.read_csv(DATA_PATH + 'Test.csv')

print('Train:', train.shape, ' | Test:', test.shape)
print('\nTarget distribution:')
print(train['bank_account'].value_counts())

# Save submission IDs before any transformation
test_ids = test['uniqueid'] + ' x ' + test['country']

# Encode target: Yes -> 1, No -> 0
train['bank_account'] = (train['bank_account'] == 'Yes').astype(int)
print('\nPositive rate:', round(train['bank_account'].mean(), 4))

Train: (23524, 13)  | Test: (10086, 12)

Target distribution:
bank_account
No     20212
Yes     3312
Name: count, dtype: int64

Positive rate: 0.1408


In [ ]:
# ── Feature engineering (v3 — NO leaky target encoding here) ──────────────
EDU_ORDER = {
    'No formal education': 0,
    'Primary education': 1,
    'Secondary education': 2,
    'Vocational/Specialised training': 3,
    'Tertiary education': 4,
    'Other/Dont know/RTA': 2,
}

def engineer_features(df):
    df = df.copy()

    # Education ordinal
    df['education_ordinal'] = df['education_level'].map(EDU_ORDER).fillna(2).astype(int)

    # Age features
    df['log_age'] = np.log1p(df['age_of_respondent'])
    df['age_bin'] = pd.cut(df['age_of_respondent'],
                           bins=[0, 20, 30, 40, 50, 60, 100],
                           labels=False).fillna(0).astype(int)

    # Household features
    df['log_household'] = np.log1p(df['household_size'])
    df['household_bin'] = pd.cut(df['household_size'],
                                  bins=[0, 2, 4, 6, 8, 100],
                                  labels=False).fillna(0).astype(int)

    # Numeric interactions (no target leakage)
    df['age_x_edu']       = df['age_of_respondent'] * df['education_ordinal']
    df['household_x_edu'] = df['household_size']    * df['education_ordinal']
    df['age_x_household'] = df['age_of_respondent'] * df['household_size']

    return df

In [ ]:
# ── Apply feature engineering (safe FE only) ───────────────────────────────
train = engineer_features(train)
test  = engineer_features(test)

# ── Categorical label encoding (fit on train + test combined) ──────────────
CAT_COLS = ['country', 'location_type', 'cellphone_access',
            'gender_of_respondent', 'relationship_with_head',
            'marital_status', 'education_level', 'job_type']

for col in CAT_COLS:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

# ── Define features ────────────────────────────────────────────────────────
DROP_COLS    = ['bank_account', 'uniqueid']
FEATURE_COLS = [c for c in train.columns if c not in DROP_COLS]

X      = train[FEATURE_COLS].copy()
y      = train['bank_account'].copy()
X_test = test[FEATURE_COLS].copy()

print('Features before TE:', len(FEATURE_COLS))
print(FEATURE_COLS)

Features before TE: 19
['country', 'year', 'location_type', 'cellphone_access', 'household_size', 'age_of_respondent', 'gender_of_respondent', 'relationship_with_head', 'marital_status', 'education_level', 'job_type', 'education_ordinal', 'log_age', 'age_bin', 'log_household', 'household_bin', 'age_x_edu', 'household_x_edu', 'age_x_household']


In [ ]:
# ── Proper Out-of-Fold Target Encoding (no leakage) ────────────────────────
def oof_target_encode(series_train, y_train, series_test, smoothing=30,
                      n_splits=5, seed=0):
    kf       = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof      = np.zeros(len(series_train))
    gm       = y_train.mean()

    series_train = series_train.reset_index(drop=True)
    y_train      = y_train.reset_index(drop=True)
    series_test  = series_test.reset_index(drop=True)

    for tr_idx, val_idx in kf.split(series_train):
        stats = pd.DataFrame({
            'c': series_train.iloc[tr_idx].values,
            'y': y_train.iloc[tr_idx].values,
        }).groupby('c')['y'].agg(['mean', 'count'])
        smoothed        = (stats['count'] * stats['mean'] +
                           smoothing * gm) / (stats['count'] + smoothing)
        oof[val_idx]    = series_train.iloc[val_idx].map(smoothed).fillna(gm).values

    stats = pd.DataFrame({
        'c': series_train.values,
        'y': y_train.values,
    }).groupby('c')['y'].agg(['mean', 'count'])
    smoothed = (stats['count'] * stats['mean'] + smoothing * gm) / (stats['count'] + smoothing)
    test_enc = series_test.map(smoothed).fillna(gm).values

    return oof, test_enc


# ── 1-way TE on high-cardinality columns only ─────────────────────────────
single_te_cols = ['country', 'job_type', 'education_level',
                  'relationship_with_head', 'marital_status', 'location_type']

for col in single_te_cols:
    tr_enc, te_enc = oof_target_encode(X[col], y, X_test[col], smoothing=30)
    X[col + '_te']      = tr_enc
    X_test[col + '_te'] = te_enc

# ── 2-way TE on meaningful pairs ──────────────────────────────────────────
pair_te = [
    ('country',          'job_type'),
    ('country',          'education_level'),
    ('country',          'location_type'),
    ('country',          'cellphone_access'),
    ('job_type',         'education_level'),
    ('job_type',         'gender_of_respondent'),
    ('cellphone_access', 'job_type'),
]

for c1, c2 in pair_te:
    combined_tr = X[c1].astype(str) + '|' + X[c2].astype(str)
    combined_te = X_test[c1].astype(str) + '|' + X_test[c2].astype(str)
    tr_enc, te_enc = oof_target_encode(combined_tr, y, combined_te, smoothing=50)
    name = c1 + '_' + c2 + '_te'
    X[name]      = tr_enc
    X_test[name] = te_enc

FEATURE_COLS = list(X.columns)
print('Features after OOF TE:', len(FEATURE_COLS))
print('New TE columns :', [c for c in FEATURE_COLS if c.endswith('_te')])

Features after OOF TE: 32
New TE columns : ['country_te', 'job_type_te', 'education_level_te', 'relationship_with_head_te', 'marital_status_te', 'location_type_te', 'country_job_type_te', 'country_education_level_te', 'country_location_type_te', 'country_cellphone_access_te', 'job_type_education_level_te', 'job_type_gender_of_respondent_te', 'cellphone_access_job_type_te']


In [ ]:
# ── LightGBM — 10-Fold Stratified CV (no scale_pos_weight) ────────────────
N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

LGBM_PARAMS = {
    'n_estimators'     : 5000,
    'learning_rate'    : 0.01,
    'num_leaves'       : 63,
    'max_depth'        : -1,
    'min_child_samples': 30,
    'feature_fraction' : 0.80,
    'bagging_fraction' : 0.80,
    'bagging_freq'     : 5,
    'reg_alpha'        : 0.10,
    'reg_lambda'       : 1.00,
    'random_state'     : 42,
    'n_jobs'           : -1,
    'verbose'          : -1,
}

oof_lgbm  = np.zeros(len(X))
pred_lgbm = np.zeros(len(X_test))

print('--- LightGBM 10-Fold CV ---')
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    mdl = LGBMClassifier(**LGBM_PARAMS)

    if LGBM_NEW_API:
        mdl.fit(X_tr, y_tr,
                eval_set=[(X_val, y_val)],
                categorical_feature=CAT_COLS,
                callbacks=[early_stopping(100, verbose=False)])
    else:
        mdl.fit(X_tr, y_tr,
                eval_set=[(X_val, y_val)],
                categorical_feature=CAT_COLS,
                early_stopping_rounds=100,
                verbose=False)

    oof_lgbm[val_idx]  = mdl.predict_proba(X_val)[:, 1]
    pred_lgbm         += mdl.predict_proba(X_test)[:, 1] / N_SPLITS

    fold_mae = 1 - accuracy_score(y_val, (oof_lgbm[val_idx] > 0.5).astype(int))
    print('  Fold', fold + 1, '| MAE =', round(fold_mae, 6))

lgbm_oof_mae = 1 - accuracy_score(y, (oof_lgbm > 0.5).astype(int))
print('\nLightGBM OOF MAE:', round(lgbm_oof_mae, 6))

--- LightGBM 10-Fold CV ---
  Fold 1 | MAE = 0.115597
  Fold 2 | MAE = 0.116447
  Fold 3 | MAE = 0.112622
  Fold 4 | MAE = 0.113047
  Fold 5 | MAE = 0.11267
  Fold 6 | MAE = 0.102041
  Fold 7 | MAE = 0.117772
  Fold 8 | MAE = 0.116071
  Fold 9 | MAE = 0.117347
  Fold 10 | MAE = 0.113095

LightGBM OOF MAE: 0.113671


In [ ]:
# ── XGBoost — 10-Fold Stratified CV (no scale_pos_weight) ─────────────────
XGB_PARAMS = {
    'n_estimators'         : 5000,
    'learning_rate'        : 0.01,
    'max_depth'            : 5,
    'min_child_weight'     : 3,
    'subsample'            : 0.80,
    'colsample_bytree'     : 0.80,
    'gamma'                : 0.10,
    'reg_alpha'            : 0.10,
    'reg_lambda'           : 2.00,
    'random_state'         : 42,
    'n_jobs'               : -1,
    'eval_metric'          : 'error',
    'early_stopping_rounds': 100,
}

oof_xgb  = np.zeros(len(X))
pred_xgb = np.zeros(len(X_test))

print('--- XGBoost 10-Fold CV ---')
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    for attempt in range(3):
        try:
            mdl = XGBClassifier(**XGB_PARAMS)
            mdl.fit(X_tr, y_tr,
                    eval_set=[(X_val, y_val)],
                    verbose=False)
            break
        except Exception as e:
            print('  Fold', fold + 1, 'attempt', attempt + 1, 'failed:', e)
            if attempt < 2:
                time.sleep(5)

    oof_xgb[val_idx]  = mdl.predict_proba(X_val)[:, 1]
    pred_xgb         += mdl.predict_proba(X_test)[:, 1] / N_SPLITS

    fold_mae = 1 - accuracy_score(y_val, (oof_xgb[val_idx] > 0.5).astype(int))
    print('  Fold', fold + 1, '| MAE =', round(fold_mae, 6))

xgb_oof_mae = 1 - accuracy_score(y, (oof_xgb > 0.5).astype(int))
print('\nXGBoost OOF MAE:', round(xgb_oof_mae, 6))

--- XGBoost 10-Fold CV ---
  Fold 1 | MAE = 0.113047
  Fold 2 | MAE = 0.112197
  Fold 3 | MAE = 0.111347
  Fold 4 | MAE = 0.109647
  Fold 5 | MAE = 0.107143
  Fold 6 | MAE = 0.10119
  Fold 7 | MAE = 0.118622
  Fold 8 | MAE = 0.115646
  Fold 9 | MAE = 0.117772
  Fold 10 | MAE = 0.110969

XGBoost OOF MAE: 0.111758


In [ ]:
# ── CatBoost — 10-Fold Stratified CV (no scale_pos_weight) ────────────────
cat_indices = [list(X.columns).index(c) for c in CAT_COLS if c in list(X.columns)]

CATBOOST_PARAMS = {
    'iterations'   : 5000,
    'learning_rate': 0.01,
    'depth'        : 6,
    'l2_leaf_reg'  : 5.0,
    'random_seed'  : 42,
    'eval_metric'  : 'Accuracy',
    'od_type'      : 'Iter',
    'od_wait'      : 100,
    'task_type'    : 'CPU',
    'verbose'      : False,
}

oof_cat  = np.zeros(len(X))
pred_cat = np.zeros(len(X_test))

print('--- CatBoost 10-Fold CV ---')
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    mdl = CatBoostClassifier(**CATBOOST_PARAMS)
    mdl.fit(X_tr, y_tr,
            cat_features=cat_indices,
            eval_set=(X_val, y_val),
            use_best_model=True,
            verbose=False)

    oof_cat[val_idx]  = mdl.predict_proba(X_val)[:, 1]
    pred_cat         += mdl.predict_proba(X_test)[:, 1] / N_SPLITS

    fold_mae = 1 - accuracy_score(y_val, (oof_cat[val_idx] > 0.5).astype(int))
    print('  Fold', fold + 1, '| MAE =', round(fold_mae, 6))

cat_oof_mae = 1 - accuracy_score(y, (oof_cat > 0.5).astype(int))
print('\nCatBoost OOF MAE:', round(cat_oof_mae, 6))

--- CatBoost 10-Fold CV ---
  Fold 1 | MAE = 0.115597
  Fold 2 | MAE = 0.115172
  Fold 3 | MAE = 0.112622
  Fold 4 | MAE = 0.108372
  Fold 5 | MAE = 0.108844
  Fold 6 | MAE = 0.103316
  Fold 7 | MAE = 0.121173
  Fold 8 | MAE = 0.114796
  Fold 9 | MAE = 0.116071
  Fold 10 | MAE = 0.108418

CatBoost OOF MAE: 0.112438


In [ ]:
# ── Blend + Threshold Optimisation + Submission ───────────────────────────
print('Individual OOF MAE (@ threshold 0.5):')
print('  LightGBM :', round(1 - accuracy_score(y, (oof_lgbm > 0.5).astype(int)), 6))
print('  XGBoost  :', round(1 - accuracy_score(y, (oof_xgb  > 0.5).astype(int)), 6))
print('  CatBoost :', round(1 - accuracy_score(y, (oof_cat  > 0.5).astype(int)), 6))

# Search best blend weights + threshold on OOF
best = {'mae': float('inf')}

for wL in np.arange(0.20, 0.71, 0.10):
    for wX in np.arange(0.10, 0.71, 0.10):
        wC = 1.0 - wL - wX
        if wC < 0.05 or wC > 0.70:
            continue
        oof_b = wL * oof_lgbm + wX * oof_xgb + wC * oof_cat
        for t in np.arange(0.30, 0.70, 0.01):
            mae = 1 - accuracy_score(y, (oof_b > t).astype(int))
            if mae < best['mae']:
                best.update({'mae': mae, 'wL': wL, 'wX': wX, 'wC': wC, 't': t})

print('\nBest blend   :',
      'wL=', round(best['wL'], 2),
      ' wX=', round(best['wX'], 2),
      ' wC=', round(best['wC'], 2))
print('Best threshold:', round(best['t'], 3))
print('Blended OOF MAE:', round(best['mae'], 6))

# Fine-tune threshold on best weights
wL, wX, wC = best['wL'], best['wX'], best['wC']
oof_blend  = wL * oof_lgbm  + wX * oof_xgb  + wC * oof_cat
pred_blend = wL * pred_lgbm + wX * pred_xgb + wC * pred_cat

best_t, best_mae = 0.50, float('inf')
for t in np.arange(0.20, 0.80, 0.002):
    mae = 1 - accuracy_score(y, (oof_blend > t).astype(int))
    if mae < best_mae:
        best_mae, best_t = mae, t

print('\n=== FINAL RESULTS ===')
print('Final OOF MAE :', round(best_mae, 6))
print('Final threshold:', round(best_t, 4))

# Final predictions & submission
final_preds = (pred_blend > best_t).astype(int)

submission = pd.DataFrame({
    'uniqueid'    : test_ids,
    'bank_account': final_preds,
})

OUT_FILE = 'enhanced_submission_v3.csv'
submission.to_csv(OUT_FILE, index=False)
print('\nSaved:', OUT_FILE)
print('Predicted bank_account = 1 :', final_preds.sum(), '/', len(final_preds),
      '(', round(final_preds.mean(), 4), ')  —  true train positive rate:',
      round(y.mean(), 4))
print(submission.head(10))

Individual OOF MAE (@ threshold 0.5):
  LightGBM : 0.113671
  XGBoost  : 0.111758
  CatBoost : 0.112438

Best blend   : wL= 0.2  wX= 0.5  wC= 0.3
Best threshold: 0.44
Blended OOF MAE: 0.111801

=== FINAL RESULTS ===
Final OOF MAE : 0.111801
Final threshold: 0.44

Saved: enhanced_submission_v3.csv
Predicted bank_account = 1 : 849 / 10086 ( 0.0842 )  —  true train positive rate: 0.1408
                uniqueid  bank_account
0  uniqueid_6056 x Kenya             1
1  uniqueid_6060 x Kenya             1
2  uniqueid_6065 x Kenya             0
3  uniqueid_6072 x Kenya             0
4  uniqueid_6073 x Kenya             0
5  uniqueid_6074 x Kenya             0
6  uniqueid_6075 x Kenya             0
7  uniqueid_6076 x Kenya             1
8  uniqueid_6077 x Kenya             0
9  uniqueid_6078 x Kenya             1
